In [1]:
# magic code
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.insert(0, ".")
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [7]:
from securerag.config import Config


cfg = Config()
cfg.device = "cpu"
cfg.n_context = 10
cfg.batch_size = 10 
cfg.private_passage_ratio = 0.5

In [4]:
# Prepare Data
import torch
import transformers
from securerag import data

checkpoint_path = "models/nq_reader_base"

path = "data/open_domain_data/NQ/debug.json"
datas = data.load(path=path, cfg=cfg)
dataset = data.Dataset(data=datas, n_context=cfg.n_context)
tokenizer: transformers.T5Tokenizer = transformers.T5Tokenizer.from_pretrained(
    "models/t5-base", return_dict=False
)
data_loader = torch.utils.data.dataloader.DataLoader(
    dataset=dataset,
    batch_size=cfg.batch_size,
    collate_fn=data.SecureRAGUsingT5EncodingCollator(
        tokenizer=tokenizer,
        text_maxlength=cfg.text_maxlength,
        answer_maxlength=cfg.answer_maxlength,
        private_passage_ratio=cfg.private_passage_ratio,
    ),
)
record1 = next(iter(data_loader))

2025-06-29 13:06:41,084	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-06-29 13:06:44,055	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1767: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_t5.py:184: UserWarning: Th

In [5]:
# Model
from securerag.models import FiDT5


model_cls = FiDT5
model_path = cfg.generator_model_path
model = model_cls.from_pretrained(model_path)
model = model.to(cfg.device)
model.eval()

#  convert to securerag from fidtt5
from securerag.models.securerag import SecureRAG

model: SecureRAG = SecureRAG(fidt5=model)

/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/modeling_utils.py:927: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(resolve

In [9]:
# generate
import time


device = cfg.device
(
    question_ids,  # bsz * 1 * dim
    question_masks,
    context_ids,
    context_masks,  # bsz * docs * dim
    private_context_ids,
    private_context_masks,  # bsz * docs_p * dim
    scores,  # bsz * docs
) = (
    record1.question_ids,
    record1.question_masks,
    record1.passage_ids,
    record1.passage_masks,
    record1.private_passage_ids,
    record1.private_passage_masks,
    record1.scores,
)

start = time.time()
output = model.generate(
    context_ids=context_ids,
    context_ids_private=private_context_ids,
    attention_mask=context_masks,
    attention_mask_private=private_context_masks,
    # TODO the dim of scores need to split
    doc_scores=scores,
    doc_scores_private=scores,
    max_length=50,
)
ans = tokenizer.batch_decode(output, skip_special_tokens=True)
print(ans)
print(f"elapsed time : {time.time() - start: .3f} sec")

['Linda Kaye Davis', '2,000', 'through the Saint Lawrence River', 'October 2019', 'r', 'Solange Knowles', 'Roy Larson Raymond', '2002', 'Parliament of the British Empire', 'Murco Records']
elapsed time :  101.103 sec
